<a href="https://colab.research.google.com/github/barbosa1991/Single_imune/blob/main/project_single_immune.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#importar o drive
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [2]:
#criando a pasta para os arquivos que serão rodados no script
import os
data_dir = '/content/drive/MyDrive/SingleCell_data'
os.makedirs(data_dir, exist_ok=True)

In [3]:
#verificar se a pasta foi criada
os.listdir('/content/drive/MyDrive/SingleCell_data')


[]

In [4]:
#verificar espaço no GoogleColab para verificar se o arquivo pode ser armazenado aqui.
!df -h /content


Filesystem      Size  Used Avail Use% Mounted on
overlay         108G   90M  108G   1% /


In [5]:
#download do arquivo direto no GoogleColab
url = "https://datasets.cellxgene.cziscience.com/688185ad-11c2-4172-a53a-f4f1f4076860.h5ad"
!wget -c --show-progress "$url" -O /content/single_cell_dataset.h5ad

--2026-11-21 23:20:52--  https://datasets.cellxgene.cziscience.com/688185ad-11c2-4172-a53a-f4f1f4076860.h5ad
Resolving datasets.cellxgene.cziscience.com (datasets.cellxgene.cziscience.com)... 13.225.143.127, 13.225.143.79, 13.225.143.98, ...
Connecting to datasets.cellxgene.cziscience.com (datasets.cellxgene.cziscience.com)|13.225.143.127|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 5873612847 (5.5G) [binary/octet-stream]
Saving to: ‘/content/single_cell_dataset.h5ad’

/content/single_cel 100%[===================>]   5.47G  31.1MB/s    in 62s     

2026-11-21 23:21:54 (89.8 MB/s) - ‘/content/single_cell_dataset.h5ad’ saved [5873612847/5873612847]



In [6]:
#copiar o arquivo para o Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
import shutil
folder = "/content/drive/MyDrive/SingleCell_data"

shutil.copy2(
    "/content/single_cell_dataset.h5ad",
    folder + "/single_cell_dataset.h5ad"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


'/content/drive/MyDrive/SingleCell_data/single_cell_dataset.h5ad'

In [8]:
import os

print(os.path.exists('/content/drive/MyDrive/'))


True


In [14]:
#installing the libraries we're gonna use.
!pip install scanpy

  Using cached anndata-0.13.4-py3-none-any.whl.metadata (7.2 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 30.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 100.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 56.0 MB/s  0:00:00
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.3
    Uninstalling pandas-2.2.3:
      Successfully uninstalled pandas-2.2.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13/13 [scanpy]
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.1.1 which is incompatible.


In [1]:
#continue installing the libraries we're gonna use.
!pip install AnnData
!pip install lamindb
!pip install numpy
!pip install pandas
!pip install seaborn
!pip install rpy2
!pip install scipy
!pip install matplotlib

  Using cached lamindb-2.12.2-py3-none-any.whl.metadata (22 kB)
INFO: pip is looking at multiple versions of boto3 to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of boto3 to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 705.4/705.4 kB 9.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 37.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.3/8.3 MB 34.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 604.6/604.6 kB 23.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 80.4 MB/s  0:00:00
  Attempting uninstall: f

O segundo passo consistirá em fazer a manipulação dos arquivos da amostra obtida e realizar uma primeira análise:
*italicized text*


In [3]:
#Import libraries before start running the script

from typing import Literal

import AnnData as ad
import lamindb as ln
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import seaborn as sns
from rpy2.robjects import numpy2ri
from rpy2.robjects.conversion import localconverter
from scipy.sparse import csc_matrix
from scipy.special import logit
from scipy.stats import median_abs_deviation


/usr/local/lib/python3.13/dist-packages/pandas/core/arrays/masked.py:63: UserWarning: Pandas requires version '1.5.0' or newer of 'bottleneck' (version '1.4.2' currently installed).
  from pandas.core import (


In [ ]:
#Load sample
adata = sc.read.h5ad("nome_do_arquivo.h5ad")

If an error appear showing that genes are duplicated, we used this function to make sure each gene name is unique, by appending a number string to each duplicate.

In [ ]:
#Make names unique
adata.var_names_make_unique()
adata


**Filtering low quality cells.**

Three variates are evaluate:
1) The number of counts per barcode (count depth)
2) The number of genes per barcode
3) The fraction of counts from mitochondrial genes per barcode.

In [ ]:
#Calculate the proportions of counts for specific gene populations: mitochondrial, ribosomal, and hemoglobin genes.
#calculate the number of mitochondrial genes
adata.var["mt"] = adata.var_names.str.startswith("MT-")

#calculate the number of ribosomal genes (step used for additional diagnostics)
adata.var["ribo"] = adata.var_names.str.startswith("RPS", "RPL")

#calculate the number of hemoglobin genes (step used for additional diagnostics)
adata.var["hb"] = adata.var_names.str.contains(r"^HB"[ABDEGMQZ]\d*?|\w")


In [ ]:
#QC metrics calculation with scanpy
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt", "ribo", "hb"], inplace=True), percent_top=20, log1p=True)
adata

We are gonna to evaluate visually how the samples are.
To do so, we construct three plots:
1) Joint plot - evaluate the distribution of two metrics at the same time.
2) Histoplot - evaluate the distribution of a metric.
3) Scatter plot - evaluate the relation between two metrics.

In [ ]:
#Plot the QC covariates to interpret how the cells were captured.
p1 = sns.jointplot(data=adata.obs, x="log1p_total_counts", y="log1p_n_genes_by_counts", kind=hex)
p2 = sns.histoplot(adata.obs["pct_counts_mt"])
p3 = sc.pi.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")


As data becomes bigger every day, we use automatic thresholding via MAD (median absolute deviation).

Cells are marked as outliers if they differ by 5 MADs. *Important to evaluate whether this thresholds is good for your analysis*.

In [ ]:
#QC with automatic thresholding
def mad_thresholds(values: pd.Series, nmad: float) -> tuple[float, float]:
  """Thresholds 'nmads' median absolute deviations below and above the median of 'values'."""
  median, mad = np.median(values), median_abs_deviation(values)
  return median - nmad * mad, median + nmad * mad

def is_outlier(
    adata: ad.AnnData,
    metrics: str,
    nmads: float,
    side: Literal["both", "upper"] = "both",
) -> pd.Series:

    """Flag the cells beyond the MAD thresholds of 'metric', on both sides or only above."""
    values = adata.obs[metrics]
    lower, upper = mad_thresholds(values, nmads)
    return values > upper if side == "upper" else (values < lower) | (values > upper)


In [ ]:
#Logit-transform mitochondrial fraction before computing MADS
mt_fraction = (adata.obs["total_counts_mt"]+1)/(adata.obs["total_counts"] + 2)
adata.obs["logit_mt_fraction"] = logit(mt_fraction)


Metrics for mitochondrial analysis.

  Cells are flagged if exceed both: 3 MADS and a fixed cutoff of 8%.

In [ ]:
#Flagging cells
qc_filters = {
    "log1p_total_counts": (5, "both"),
    "log1p_n_genes_by_counts": (5, "both"),
    "pct_counts_in_top_20_genes": (5, "both"),
    "logit_mt_fraction": (3, "upper"),
}
flags = {
    metric: is_outlier(adata, metric, nmads, side) for metric, (nmads, side) in qc_filters.items()
}
adata.obs["outlier"] = (
    flags["log1p_total_counts"]
    | flags["log1p_n_genes_by_counts"]
    | flags["pct_counts_in_top_20_genes"]
)
adata.obs["mt_outlier"] = flags["logit_mt_fraction"] & (adata.obs["pct_counts_mt"] > 8)
adata.obs["outlier", "mt_outlier"].value_counts()

Checking the thresholds is an important step to evaluate whether the range is well defined for your data.
This step is done by ploting histograms of your different variables, such as "total_counts", "log1p_n_genes_by_counts", "pct_counts_in_top_20_genes", and "logit_mt_fraction".

In [ ]:
#Checking the thresholds
fig, axes = plt.subplots(1, len(qc_filters), figsize=(16, 3.5))
for ax, (metric, (nmads, side)) in zip(axes, qc_filters.items(), strict=True):
    sns.hitoplot(adata.obs[metric], bins=100, ax=xx)
    lower, upper = mad_thresholds(adata.obs[metric], nmads)
    for threshold in [upper] if side == "upper" else [lower, upper]:
      ax.axvline(threshold, color="red", linestyle="--")
    plt.tight_layout()
    plt.show()

After confirming that the selected thresholds are good, you filter your data, and plot the selected cell using a scatter plot.

In [ ]:
#Filtering our AnnData object based on these the thresholds
print(f"Total number of cells: {adata.n_obs}")
adata = adata[(~adata.obs.outlier) & (~adata.obs.mt_outlier)].copy()
print(f"Number of cells after filtering of low quality cells: {adata.n_obs}"]


In [ ]:
#Final graph of selected data
p1 = sc.pl.scatter(adata, "total_counts", "n_genes_by_counts", color="pct_counts_mt")
